In [22]:
from email.mime import image

import requests
from adodbapi.examples.xls_read import filename
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

DOMAIN = "https://www.nikkei.com/"
page_start = 1
page_end = 2
titles = []
body_texts = []
img_urls = []

for page in range(page_start, page_end+1):
    #htmlの取得
    url = "https://www.nikkei.com/business/net-media/?page={}".format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    #データー取得-タイトル
    links= soup.find_all("a", class_="fauxBlockLink_f6t5v6i")
    #１ページ目のトップ部分はクラスが違うため、別途取得
    if page == 1:
        link_page_one = soup.find_all("a", class_="fauxBlockLink_fzas0ps")
        links.insert(0, link_page_one[0])
    for link in links:
    #U3000除く
        titles.append(link.get_text().replace("\u3000", ""))
    #本文取得



    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    for link in links:
        href = link.get("href")

        if not href:
            body_texts.append("")
            continue

        body_link = urljoin(DOMAIN, href)

        body_response = requests.get(body_link, headers=headers)
        body_soup = BeautifulSoup(body_response.text, "html.parser")

        paragraphs = body_soup.find_all("p", class_="paragraph_p18mfke4")

        if not paragraphs:
            print("本文が見つかりません:", body_link)
            body_texts.append("")
            continue

        body_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
        body_texts.append(body_text)
     #画像取得
    img_class = "image_i1obkbgm"
    img_tags = soup.find_all(class_=img_class)

    #img_urls = []
    # for img_tag in img_tags:
    #     img_url = img_tag.get("src")
    #     if not img_url:
    #       img_urls.append("")
    #       continue
    #     #if img_url:
    #       img_urls.append(img_url)
    for link in links:
        article = link.find_parent()
        img_tag = article.find("img") if article else None

        if img_tag and img_tag.get("src"):
            img_urls.append(img_tag.get("src"))
        else:
            img_urls.append("")
news_date = {
    'title': titles,
    'body_texts': body_texts,
    'img_urls': img_urls
}

# print(len(titles))
# print(len(body_texts))
# print(len(img_urls))
df = pd.DataFrame.from_dict(news_date)
df

df.to_csv("nikkei_news2.csv", index=None, encoding="utf-8-sig")






本文が見つかりません: https://www.nikkei.com/article/DGXZQOCD2826H0Y6A420C2000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC20A220Q6A420C2000000/


,title,body_texts,img_urls
0,ウェザーニューズ、20万人の「天気愛」をお茶の間に,天候不順や激甚災害が増えるなか、天気情報への関心はこれまでになく高まっている。高まる関心を生...,/.resources/k-components/icon/locked_square.re...
1,サイバーエージェントが報酬月50万円のインターンシップ博士後期対象,サイバーエージェントはスポーツ領域の専門組織「Sports AI Tech Lab」で、大学...,
2,みずほが楽天銀行出資へ5〜10%軸、楽天子会社再編で資本付け替え,みずほフィナンシャルグループは楽天銀行に出資する。5〜10%を軸に出資比率を詰めている。楽天...,/.resources/k-components/icon/locked_square.re...
3,「AI吹き替え」で感情や声質も再現200言語に対応、新興がサービス,人工知能（AI）を活用した音声合成を手掛けるタイタンインテリジェンス（東京・渋谷）は、話者の...,/.resources/k-components/icon/locked_square.re...
4,宣材動画の制作10分でソフトバンク、生成AIで作成代行,ソフトバンクは広告に使うショート動画の作成を生成AI（人工知能）が代行する技術を開発した。静...,/.resources/k-components/logo/think.rev-3b943d...
5,TSMC熊本工場、初の最終黒字26年1〜3月期,台湾積体電路製造（TSMC）の熊本工場を運営する子会社の四半期ベースの最終損益が量産開始後、...,
6,Claude MythosだけじゃないAIサイバー攻撃が暴く未知のバグ,米グーグルは12日、犯罪組織がAIを駆使することで開発元すら知らないソフトウエアの欠陥を発見...,/.resources/k-components/icon/locked_square.re...
7,漫画家やイラストレーター「AI普及で収入減」2割著作権保護求める声,,/.resources/k-components/logo/think.rev-3b943d...
8,ソフトバンク系､炒め物特化の調理ロボ冷蔵庫から自動で食材,ソフトバンクグループ（SBG）傘下のソフトバンクロボティクス（東京・港）は、炒め調理に特化し...,/.resources/k-components/icon/locked_square.re...
9,GMO純利益14%増1〜3月、キャッシュレス決済やネット金融好調,GMOインターネットグループが15日発表した2026年1〜3月期の連結決算（国際会計基準）は...,


In [23]:
df.to_csv("nikkei_news2.csv", index=None, encoding="utf-8-sig")

In [26]:
#画像保存
for index, row in df.iterrows():
    image_url = row["img_urls"]
    filename = str(index)
    image = requests.get(image_url)
    with open("img/"+filename+".jpg", "wb") as f:
            f.write(image.content)

MissingSchema: Invalid URL '/.resources/k-components/icon/locked_square.rev-5fc2936.svg': No scheme supplied. Perhaps you meant https:///.resources/k-components/icon/locked_square.rev-5fc2936.svg?

 ポイントは、if not img_url では「srcがない画像タグ」しか処理できないことです。
  今回の問題は、img_tag は存在するけど、それが記事画像ではなく鍵アイコンだった、という点です


原因は、img_urls に記事画像ではなく、サイトUI用のアイコンが入っていることです。
  requests.get() 側の問題ではなく、画像URLの取得段階で違う画像を拾っています。

  特にこれは記事画像ではありません。

  /.resources/k-components/icon/locked_square.rev-5fc2936.svg

  日経の「鍵アイコン」です。
